# ExoData Challenge - Fase 2: Feature Engineering & Scientific Imputation
## Tarea 2.1: Crear features derivados con significado físico

**Objetivo:** Transformar variables crudas de la NASA en features útiles para clustering,
imputando valores faltantes con relaciones físicas reales (NO con medias estadísticas).

In [1]:
# CELDA 1 — Imports y setup
import pandas as pd
import numpy as np
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from features import engineer_all_features, CLUSTERING_FEATURES
from imputation import run_scientific_imputation

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir: {DATA_DIR}')
print(f'Processed dir: {PROCESSED_DIR}')

Project root: /home/chrisriardo/datathon/ExoDataChallenge
Data dir: /home/chrisriardo/datathon/ExoDataChallenge/data
Processed dir: /home/chrisriardo/datathon/ExoDataChallenge/data/processed


In [2]:
# CELDA 2 — Cargar dataset crudo
df = pd.read_csv(DATA_DIR / 'PSCompPars_2026.csv', comment='#', low_memory=False)
print(f'Loaded: {df.shape[0]:,} planets × {df.shape[1]} columns')

Loaded: 6,160 planets × 320 columns


## Step 1: Scientific Imputation

**NO usamos fillna con la media.** Usamos relaciones físicas reales:

- **Masa ↔ Radio:** Relación M-R de Zeng & Sasselov (2016)
  - Rocoso (R < 1.5 R⊕): M ∝ R^(1/0.27)
  - Transición (1.5-4 R⊕): exponente intermedio
  - Gigante (R ≥ 4 R⊕): exponente ajustado
- **Temperatura de equilibrio:** Derivada de T_eff estelar y distancia orbital
- **Insolación:** Derivada de luminosidad estelar y distancia
- **Metalicidad:** Rellenada con valor solar (0.0) como default conservador

**Cada valor imputado se marca con un flag `_imputed` para trazabilidad.**

In [3]:
# CELDA 3 — Imputación científica
df = run_scientific_imputation(df)

Scientific Imputation Report:
----------------------------------------
  Planets missing mass (with known radius): 24
  Planets missing radius (with known mass): 43
  Planets missing insolation (estimable): 1321
  Planets missing equilibrium temp (estimable): 1098
  Planets missing metallicity: 578
  pl_bmasse_imputed: 24 values imputed
  pl_rade_imputed: 43 values imputed
  pl_insol_imputed: 1321 values imputed
  pl_eqt_imputed: 1098 values imputed
  st_met_imputed: 782 values imputed
----------------------------------------


In [4]:
# CELDA 4 — Ver resumen de imputación
imp_cols = [c for c in df.columns if c.endswith('_imputed')]
print('=== IMPUTATION SUMMARY ===')
for c in imp_cols:
    n = int(df[c].sum())
    print(f'  {c}: {n} planets ({n/len(df)*100:.1f}%)')

=== IMPUTATION SUMMARY ===
  pl_bmasse_imputed: 24 planets (0.4%)
  pl_rade_imputed: 43 planets (0.7%)
  pl_insol_imputed: 1321 planets (21.4%)
  pl_eqt_imputed: 1098 planets (17.8%)
  st_met_imputed: 782 planets (12.7%)


## Step 2: Feature Engineering

Creamos features derivados con significado astrofísico:

| Feature | Fórmula | Propósito |
|---------|---------|-----------|
| `log_period` | log10(pl_orbper) | Comprimir distribución power-law |
| `log_rade` | log10(pl_rade) | Normalizar radios |
| `log_bmasse` | log10(pl_bmasse) | Normalizar masas |
| `log_smax` | log10(pl_orbsmax) | Comprimir distancias orbitales |
| `pl_dens_calc` | ρ_earth × M/R³ | Densidad en g/cm³ — composición |
| `mass_radius_ratio` | M/R | Proxy rápido de composición |
| `kepler_check` | T²/a³ consistency | Validación de datos orbitales |
| `hz_flag` | 0.2 ≤ insol ≤ 1.7 + R<2 + 200K<T<320K | Zona habitable conservadora |

In [5]:
# CELDA 5 — Feature engineering completo
df = engineer_all_features(df)
print(f'Features añadidos. Total columnas: {len(df.columns)}')

Features añadidos. Total columnas: 334


In [6]:
# CELDA 6 — Completitud de features finales para clustering
print(f'=== COMPLETITUD DE {len(CLUSTERING_FEATURES)} FEATURES PARA CLUSTERING ===')
for f in CLUSTERING_FEATURES:
    if f in df.columns:
        pct = df[f].notna().mean() * 100
        bar = '█' * int(pct/5) + '░' * (20 - int(pct/5))
        print(f'  {f:<22} {bar} {pct:>5.1f}%')
    else:
        print(f'  {f:<22} ⚠️ MISSING')

feature_mask = df[CLUSTERING_FEATURES].notna().all(axis=1)
print(f'\nPlanetas con TODOS los features: {feature_mask.sum():,} / {len(df):,} ({feature_mask.mean()*100:.1f}%)')

=== COMPLETITUD DE 10 FEATURES PARA CLUSTERING ===
  log_rade               ███████████████████░  99.7%
  log_bmasse             ███████████████████░  99.7%
  log_period             ██████████████████░░  94.4%
  log_smax               ██████████████████░░  94.7%
  pl_dens_calc           ███████████████████░  99.7%
  mass_radius_ratio      ███████████████████░  99.7%
  pl_eqt                 ██████████████████░░  92.3%
  st_teff                ██████████████████░░  94.9%
  st_met                 ████████████████████ 100.0%


  sy_pnum                ████████████████████ 100.0%



Planetas con TODOS los features: 5,469 / 6,160 (88.8%)


In [7]:
# CELDA 7 — Estadísticas descriptivas de features
df[CLUSTERING_FEATURES].describe()

,log_rade,log_bmasse,log_period,log_smax,pl_dens_calc,mass_radius_ratio,pl_eqt,st_teff,st_met,sy_pnum
count,6142.000000,6142.000000,5816.000000,5834.000000,6142.000000,6142.000000,5685.000000,5846.000000,6160.000000,6160.000000
mean,0.582238,1.390293,1.258425,-0.781673,13.281652,37.865578,1997.296032,5396.114993,0.014387,1.772727
std,0.406181,1.063309,0.917186,0.753827,261.093632,120.266018,3654.475171,1749.664706,0.179451,1.156655
min,-0.508919,-1.698970,-1.042363,-2.356547,0.030233,0.057522,20.000000,415.000000,-1.000000,1.000000
25%,0.262451,0.626340,0.637623,-1.281498,1.351790,2.215054,623.000000,4901.500000,-0.070000,1.000000
50%,0.456366,0.969416,1.045686,-0.991038,2.643105,3.174164,946.000000,5545.000000,0.000000,1.000000
75%,1.075655,2.296495,1.597778,-0.511344,4.761960,16.142792,1569.700000,5899.750000,0.120000,2.000000
max,1.940546,4.585508,8.604226,4.278754,13870.881274,3685.727273,104323.000000,57000.000000,0.790000,8.000000


In [8]:
# CELDA 8 — Candidatos a zona habitable
hz = df[df['hz_flag'] == 1]
print(f'=== ZONA HABITABLE CONSERVADORA ===')
print(f'Candidatos: {len(hz)} ({len(hz)/len(df)*100:.2f}% del total)')
print()
print('Criterios: 0.2 ≤ insolación ≤ 1.7, radio < 2 R⊕, 200K ≤ T_eq ≤ 320K')
print()
display(hz[['pl_name', 'pl_rade', 'pl_insol', 'pl_eqt', 'st_teff', 'pl_orbper']].head(15))

=== ZONA HABITABLE CONSERVADORA ===
Candidatos: 34 (0.55% del total)

Criterios: 0.2 ≤ insolación ≤ 1.7, radio < 2 R⊕, 200K ≤ T_eq ≤ 320K



,pl_name,pl_rade,pl_insol,pl_eqt,st_teff,pl_orbper
211,GJ 1002 b,1.030,0.670000,230.90,3024.0,10.346500
244,GJ 251 c,1.800,0.404000,216.00,3342.0,53.647000
372,Gliese 12 b,0.930,1.620000,314.60,3328.0,12.761418
1693,K2-288 B b,1.900,0.425606,226.36,3341.0,31.393463
1705,K2-3 d,1.458,1.440000,305.20,3844.0,44.556030
2450,Kepler-1229 b,1.400,0.524000,212.00,3784.0,86.828989
2702,Kepler-1410 b,1.780,1.060000,290.00,4092.0,60.866168
2888,Kepler-1544 b,1.780,0.833000,269.00,4886.0,168.811174
3023,Kepler-1649 c,1.060,0.750000,234.00,3240.0,19.535270
3028,Kepler-1652 b,1.600,0.810000,268.00,3638.0,38.097220


In [9]:
# CELDA 9 — Guardar dataset procesado
df.to_csv(PROCESSED_DIR / 'df_clean_features.csv', index=False)
print(f'✓ Dataset completo guardado: {PROCESSED_DIR / "df_clean_features.csv"}')

# Guardar matriz de features para clustering (solo planetas con features completos)
X = df[feature_mask][CLUSTERING_FEATURES].copy()
X.to_csv(PROCESSED_DIR / 'X_features.csv', index=False)
print(f'✓ Matriz de features guardada: {PROCESSED_DIR / "X_features.csv"} ({X.shape[0]} filas × {X.shape[1]} columnas)')
print(f'\n✓✓✓ FASE 2 COMPLETA — {X.shape[0]} planetas listos para clustering ✓✓✓')

✓ Dataset completo guardado: /home/chrisriardo/datathon/ExoDataChallenge/data/processed/df_clean_features.csv
✓ Matriz de features guardada: /home/chrisriardo/datathon/ExoDataChallenge/data/processed/X_features.csv (5469 filas × 10 columnas)

✓✓✓ FASE 2 COMPLETA — 5469 planetas listos para clustering ✓✓✓
